# Prithvi-100M — Wildfire Burn Scar Segmentation

**Environment**: Colab A100 (GPU required)

End-to-end notebook: training, threshold optimization, and continuation fine-tuning.

---

## Sections

| Cells | Section | Description |
|---|---|---|
| 1–2 | Setup | Install, Drive mount |
| 3–4 | Imports & Paths | All constants and hyperparameters |
| 5–6 | Architecture | Dataset class, PrithviNeck + FPNDecoder |
| 7–8 | Load backbone | Prithvi-EO-1.0-100M frozen |
| 9–11 | **Train v1.0** | Epochs 1–40, loss + training loop + curves |
| 12–13 | **Threshold sweep (v1.1)** | Optimal threshold 0.50→0.65 |
| 14–16 | **Continue training (v1.2)** | Epochs 41–80 |
| 17 | Portfolio figure | Best/Median/Worst + metrics |

> **Quick eval only** (skip training): run 1–8 → 12 → 13 → 17

In [ ]:
# [1] Install dependencies
import subprocess, sys, os

needs = False
try:
    import numpy as np
    assert tuple(int(x) for x in np.__version__.split('.')[:2]) >= (2, 0)
    from terratorch.registry import BACKBONE_REGISTRY
    print(f'OK — numpy={np.__version__}, terratorch ready.')
except Exception as e:
    needs = True
    print(f'Installing ({e})...')

if needs:
    subprocess.run([sys.executable,'-m','pip','install','-q','numpy==2.0.2','--force-reinstall'], check=True)
    subprocess.run([sys.executable,'-m','pip','install','-q',
                    'terratorch','einops','timm','segmentation-models-pytorch'], check=True)
    print('Restarting kernel...')
    os.kill(os.getpid(), 9)


In [ ]:
# [2] Drive mount & environment
try:
    import google.colab; IN_COLAB = True
    from google.colab import drive; drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
print(f'IN_COLAB = {IN_COLAB}')


In [ ]:
# [3] Imports
import os, random, subprocess, warnings, time
from pathlib import Path
from tqdm import tqdm

import numpy as np
import torch
import torch.nn as nn
import torchvision.transforms.functional as TF
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'torch={torch.__version__}  device={DEVICE}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# [4] Paths & constants
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/GeoAI/wildfire-spread')
else:
    BASE = Path('G:/Mon Drive/GeoAI/wildfire-spread')

CKPT    = BASE / 'models/best_prithvi_burn_scar_wildfire.pth'
RESULTS = BASE / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

if IN_COLAB:
    LOCAL = Path('/content/patches')
    LOCAL.mkdir(exist_ok=True)
    if not (LOCAL / 'images').exists():
        print('Copying patches to /content/ ...')
        subprocess.run(['cp','-r', str(BASE/'data/patches/images'),     str(LOCAL)], check=True)
        subprocess.run(['cp','-r', str(BASE/'data/patches/masks_dnbr'), str(LOCAL)], check=True)
        print('Done.')
    IMG_DIR  = LOCAL / 'images'
    MASK_DIR = LOCAL / 'masks_dnbr'
else:
    IMG_DIR  = BASE / 'data/patches/images'
    MASK_DIR = BASE / 'data/patches/masks_dnbr'

# Prithvi normalization — HLS pretraining stats (B02,B03,B04,B8A,B11,B12)
BAND_IDX     = [0, 1, 2, 4, 5, 6]
PRITHVI_MEAN = np.array([0.033349,0.057012,0.058897,0.232325,0.197285,0.119449], dtype=np.float32)
PRITHVI_STD  = np.array([0.022691,0.026808,0.040041,0.077917,0.087087,0.072420], dtype=np.float32)
IMG_SIZE  = 224
T         = (256 - IMG_SIZE) // 2
VAL_FRAC  = 0.2
BATCH     = 8
FIRE_W    = 5.0

img_paths  = sorted(IMG_DIR.glob('*.npy'))
mask_paths = sorted(MASK_DIR.glob('*.npy'))
print(f'Patches: {len(img_paths)}')


In [ ]:
# [5] Dataset class
class BurnScarDataset(Dataset):
    def __init__(self, img_paths, mask_paths, augment=False):
        self.imgs, self.masks, self.augment = img_paths, mask_paths, augment

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        raw  = np.load(self.imgs[idx],  mmap_mode='r').astype(np.float32)
        mask = np.load(self.masks[idx], mmap_mode='r')
        img  = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
        img  = img[:,  T:T+IMG_SIZE, T:T+IMG_SIZE]
        mask = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy().astype(np.float32)
        img_t, mask_t = torch.from_numpy(img), torch.from_numpy(mask).unsqueeze(0)
        if self.augment:
            if torch.rand(1)>0.5: img_t,mask_t = TF.hflip(img_t),TF.hflip(mask_t)
            if torch.rand(1)>0.5: img_t,mask_t = TF.vflip(img_t),TF.vflip(mask_t)
            k = torch.randint(0,4,(1,)).item()
            if k: img_t,mask_t = TF.rotate(img_t,90*k),TF.rotate(mask_t,90*k)
        return img_t, mask_t

print('Dataset class defined.')


In [ ]:
# [6] Architecture — PrithviNeck + FPNDecoder
class PrithviNeck(nn.Module):
    def __init__(self, n_side=14):
        super().__init__(); self.n_side=n_side
    def forward(self, layers_out):
        x = layers_out[-1][:,1:,:]
        B,L,D = x.shape
        return x.permute(0,2,1).reshape(B,D,self.n_side,self.n_side)

class FPNDecoder(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.up = nn.Sequential(
            nn.ConvTranspose2d(in_ch,256,2,stride=2),nn.BatchNorm2d(256),nn.GELU(),
            nn.ConvTranspose2d(256,128,2,stride=2), nn.BatchNorm2d(128),nn.GELU(),
            nn.ConvTranspose2d(128,64,2,stride=2),  nn.BatchNorm2d(64), nn.GELU(),
            nn.ConvTranspose2d(64,32,2,stride=2),   nn.BatchNorm2d(32), nn.GELU(),
            nn.Conv2d(32,1,1),
        )
    def forward(self, x): return self.up(x)

class PrithviSegModel(nn.Module):
    def __init__(self, backbone, embed_dim=768, n_side=14):
        super().__init__()
        self.backbone = backbone
        self.neck     = PrithviNeck(n_side)
        self.decoder  = FPNDecoder(embed_dim)
    def forward(self, x):
        return self.decoder(self.neck(self.backbone(x.unsqueeze(2))))

print('Architecture defined.')


In [ ]:
# [7] Load Prithvi backbone (frozen)
from terratorch.registry import BACKBONE_REGISTRY

backbone = BACKBONE_REGISTRY.build('prithvi_eo_v1_100').to(DEVICE)
for p in backbone.parameters(): p.requires_grad = False

model = PrithviSegModel(backbone).to(DEVICE)
print(f'Backbone: frozen ({sum(p.numel() for p in backbone.parameters())/1e6:.0f}M params)')
print(f'Decoder : trainable ({sum(p.numel() for p in model.decoder.parameters())/1e6:.1f}M params)')


In [ ]:
# [8] Train/Val split & DataLoaders
print('Scanning fire flags...')
fire_flags = np.array([np.load(p,mmap_mode='r').max()>0 for p in tqdm(mask_paths)], dtype=np.int32)

indices = np.arange(len(img_paths))
train_idx, val_idx = train_test_split(indices, test_size=VAL_FRAC, stratify=fire_flags, random_state=SEED)

val_imgs   = [img_paths[i]  for i in val_idx]
val_masks  = [mask_paths[i] for i in val_idx]
train_imgs  = [img_paths[i]  for i in train_idx]
train_masks = [mask_paths[i] for i in train_idx]

w = torch.tensor([FIRE_W if f else 1.0 for f in fire_flags[train_idx]], dtype=torch.float)
sampler      = WeightedRandomSampler(w, len(w), replacement=True)
train_loader = DataLoader(BurnScarDataset(train_imgs,train_masks,augment=True),
                          batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(BurnScarDataset(val_imgs,val_masks,augment=False),
                          batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'Train: {len(train_idx)} patches ({fire_flags[train_idx].sum()} fire)  |  {len(train_loader)} batches')
print(f'Val  : {len(val_idx)}  patches ({fire_flags[val_idx].sum()} fire)  |  {len(val_loader)} batches')


---

## Training v1.0 — Epochs 1–40

FPN decoder trained from scratch. Prithvi backbone stays frozen throughout.

In [ ]:
# [9] Loss, optimizer & scheduler — v1.0 (epochs 1-40)
dice_loss  = smp.losses.DiceLoss(mode='binary')
focal_loss = smp.losses.FocalLoss(mode='binary')

def criterion(logits, targets):
    return dice_loss(logits, targets) + focal_loss(logits, targets)

LR_V10    = 1e-3
EPOCHS_V10 = 40

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_V10, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_V10, eta_min=1e-6)

print(f'Trainable: {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M params')
print(f'LR: {LR_V10} → 1e-6 cosine over {EPOCHS_V10} epochs')


In [ ]:
# [10] Training loop — epochs 1-40 (v1.0)
history_v10 = {'train_loss':[], 'val_loss':[], 'val_iou':[]}
best_iou_v10 = 0.0

for epoch in range(1, EPOCHS_V10 + 1):
    t0 = time.time()
    model.train(); model.backbone.eval()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = tp_s = fp_s = fn_s = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            val_loss += criterion(logits, masks).item()
            preds = (logits.sigmoid() > 0.5).long()
            t = masks.long()
            tp_s += int(((preds==1)&(t==1)).sum())
            fp_s += int(((preds==1)&(t==0)).sum())
            fn_s += int(((preds==0)&(t==1)).sum())
    val_loss /= len(val_loader)
    val_iou   = tp_s / (tp_s + fp_s + fn_s + 1e-6)

    if val_iou > best_iou_v10:
        best_iou_v10 = val_iou
        torch.save(model.state_dict(), CKPT)
        flag = ' ✓'
    else: flag = ''

    history_v10['train_loss'].append(train_loss)
    history_v10['val_loss'].append(val_loss)
    history_v10['val_iou'].append(val_iou)
    scheduler.step()
    print(f'Epoch {epoch:3d} | train={train_loss:.4f} val={val_loss:.4f} IoU={val_iou:.4f} '
          f'lr={scheduler.get_last_lr()[0]:.2e} t={time.time()-t0:.0f}s{flag}')

np.save(RESULTS/'training_history_v10.npy', history_v10)
print(f'Best val IoU v1.0: {best_iou_v10:.4f}')


In [ ]:
# [11] Training curves — v1.0
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = range(1, len(history_v10['val_iou'])+1)
axes[0].plot(ep, history_v10['train_loss'], label='Train', color='steelblue', lw=2)
axes[0].plot(ep, history_v10['val_loss'],   label='Val',   color='darkorange', lw=2)
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss — v1.0 (epochs 1-40)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history_v10['val_iou'], color='green', lw=2)
axes[1].axhline(best_iou_v10, color='red', ls='--', lw=1.5, label=f'Best: {best_iou_v10:.4f}')
axes[1].set(xlabel='Epoch', ylabel='Val IoU', title='Val IoU — v1.0 (t=0.50)')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Prithvi-100M + FPN — Training v1.0 | Corrientes', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS/'training_curves_prithvi_burn_scar.png', dpi=150, bbox_inches='tight')
plt.show()


---

## Threshold Optimization — v1.1

Default threshold 0.50 → optimal 0.65. No retraining required.

In [ ]:
# [12] Load best checkpoint & run inference on val set
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

print('Running inference...')
all_probs, all_targets = [], []
with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')
        img  = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()
        prob = model(torch.from_numpy(img).unsqueeze(0).to(DEVICE)).sigmoid().squeeze().cpu().numpy()
        all_probs.append(prob.flatten())
        all_targets.append(mc.flatten())

all_probs   = np.concatenate(all_probs).astype(np.float32)
all_targets = np.concatenate(all_targets).astype(np.uint8)
np.save(RESULTS/'val_probs.npy',   all_probs)
np.save(RESULTS/'val_labels.npy',  all_targets)
print('Saved val_probs.npy + val_labels.npy')


In [ ]:
# [13] Threshold sweep 0.05→0.95
thresholds = np.arange(0.05, 0.96, 0.025)
results    = []
for t in thresholds:
    preds = (all_probs > t).astype(np.int32)
    tp = int(((preds==1)&(all_targets==1)).sum())
    fp = int(((preds==1)&(all_targets==0)).sum())
    fn = int(((preds==0)&(all_targets==1)).sum())
    iou  = tp/(tp+fp+fn+1e-6)
    prec = tp/(tp+fp+1e-6)
    rec  = tp/(tp+fn+1e-6)
    f1   = 2*prec*rec/(prec+rec+1e-6)
    results.append((t,iou,f1,prec,rec))

results = np.array(results)
bi = results[:,2].argmax()
b  = results[np.abs(results[:,0]-0.5).argmin()]
print(f'Baseline  t=0.500: IoU={b[1]:.4f} F1={b[2]:.4f} Prec={b[3]:.4f} Rec={b[4]:.4f}')
print(f'Best F1   t={results[bi,0]:.3f}: IoU={results[bi,1]:.4f} F1={results[bi,2]:.4f} '
      f'Prec={results[bi,3]:.4f} Rec={results[bi,4]:.4f}')

THRESHOLD_OPT = float(results[bi, 0])  # 0.65

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(results[:,0],results[:,1],label='IoU',color='steelblue',lw=2)
axes[0].plot(results[:,0],results[:,2],label='F1',color='green',lw=2)
axes[0].plot(results[:,0],results[:,3],label='Precision',color='darkorange',lw=2,ls='--')
axes[0].plot(results[:,0],results[:,4],label='Recall',color='crimson',lw=2,ls='--')
axes[0].axvline(0.5,color='gray',ls=':',lw=1.5,label='t=0.50')
axes[0].axvline(results[bi,0],color='green',ls='--',lw=1.5,label=f'Best F1 (t={results[bi,0]:.2f})')
axes[0].set(xlabel='Threshold',ylabel='Score',title='Metrics vs Threshold')
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
axes[1].plot(results[:,4],results[:,3],color='steelblue',lw=2)
axes[1].scatter([b[4]],[b[3]],color='gray',s=80,zorder=5,label='t=0.50')
axes[1].scatter([results[bi,4]],[results[bi,3]],color='green',s=80,zorder=5,label=f't={results[bi,0]:.2f}')
axes[1].set(xlabel='Recall',ylabel='Precision',title='Precision-Recall Curve')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
plt.suptitle('Threshold Optimization — Prithvi-100M + FPN | Corrientes val set',fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS/'threshold_sweep.png',dpi=150,bbox_inches='tight')
plt.show()


---

## Continuation Training — v1.2 (Epochs 41–80)

Second cosine LR cycle at lower rate (5e-5→1e-6). Backbone stays frozen. Uses optimal threshold from v1.1.

In [ ]:
# [14] Loss, optimizer & scheduler — v1.2 (epochs 41-80)
LR_V12      = 5e-5
EPOCHS_V12  = 40
START_EPOCH = 41

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR_V12, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_V12, eta_min=1e-6)
print(f'LR: {LR_V12} → 1e-6 cosine over {EPOCHS_V12} epochs')
print(f'Decision threshold: {THRESHOLD_OPT}')


In [ ]:
# [15] Training loop — epochs 41-80 (v1.2)
history_v12 = {'train_loss':[], 'val_loss':[], 'val_iou':[]}
best_iou_v12 = 0.0

for epoch in range(START_EPOCH, START_EPOCH + EPOCHS_V12):
    t0 = time.time()
    model.train(); model.backbone.eval()
    train_loss = 0.0
    for imgs, masks in train_loader:
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward(); optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    model.eval()
    val_loss = tp_s = fp_s = fn_s = 0
    with torch.no_grad():
        for imgs, masks in val_loader:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            logits = model(imgs)
            val_loss += criterion(logits, masks).item()
            preds = (logits.sigmoid() > THRESHOLD_OPT).long()
            t = masks.long()
            tp_s += int(((preds==1)&(t==1)).sum())
            fp_s += int(((preds==1)&(t==0)).sum())
            fn_s += int(((preds==0)&(t==1)).sum())
    val_loss /= len(val_loader)
    val_iou   = tp_s / (tp_s + fp_s + fn_s + 1e-6)

    if val_iou > best_iou_v12:
        best_iou_v12 = val_iou
        torch.save(model.state_dict(), CKPT)
        flag = ' ✓'
    else: flag = ''

    history_v12['train_loss'].append(train_loss)
    history_v12['val_loss'].append(val_loss)
    history_v12['val_iou'].append(val_iou)
    scheduler.step()
    print(f'Epoch {epoch:3d} | train={train_loss:.4f} val={val_loss:.4f} IoU={val_iou:.4f} '
          f'lr={scheduler.get_last_lr()[0]:.2e} t={time.time()-t0:.0f}s{flag}')

np.save(RESULTS/'training_history_v12.npy', history_v12)
print(f'Best val IoU v1.2: {best_iou_v12:.4f}')


In [ ]:
# [16] Training curves — v1.2
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ep = list(range(START_EPOCH, START_EPOCH+len(history_v12['val_iou'])))
axes[0].plot(ep, history_v12['train_loss'], label='Train', color='steelblue', lw=2)
axes[0].plot(ep, history_v12['val_loss'],   label='Val',   color='darkorange', lw=2)
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Loss — v1.2 (epochs 41-80)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[1].plot(ep, history_v12['val_iou'], color='green', lw=2)
axes[1].axhline(best_iou_v12, color='red', ls='--', lw=1.5, label=f'Best: {best_iou_v12:.4f}')
axes[1].set(xlabel='Epoch', ylabel='Val IoU', title=f'Val IoU — v1.2 (t={THRESHOLD_OPT})')
axes[1].legend(); axes[1].grid(alpha=0.3)
plt.suptitle('Prithvi-100M + FPN — Continuation Training v1.2 | Corrientes', fontsize=11)
plt.tight_layout()
plt.savefig(RESULTS/'training_curves_continuation.png', dpi=150, bbox_inches='tight')
plt.show()


---

## Portfolio Figure

Best/Median/Worst patches + training convergence + global metrics.

In [ ]:
# [17] Portfolio figure — validation_overview.png
model.load_state_dict(torch.load(CKPT, map_location=DEVICE))
model.eval()

print('Running inference for figure...')
all_samples = []
tp_all = fp_all = fn_all = 0
with torch.no_grad():
    for ip, mp in tqdm(zip(val_imgs, val_masks), total=len(val_imgs)):
        raw  = np.load(ip,  mmap_mode='r').astype(np.float32)
        mask = np.load(mp,  mmap_mode='r')
        img  = (raw[BAND_IDX]/10000.0 - PRITHVI_MEAN[:,None,None]) / PRITHVI_STD[:,None,None]
        img  = img[:, T:T+IMG_SIZE, T:T+IMG_SIZE]
        mc   = mask[T:T+IMG_SIZE, T:T+IMG_SIZE].copy()
        prob = model(torch.from_numpy(img).unsqueeze(0).to(DEVICE)).sigmoid().squeeze().cpu().numpy()
        pred = (prob > THRESHOLD_OPT).astype(np.uint8)
        tp = int(((pred==1)&(mc==1)).sum())
        fp = int(((pred==1)&(mc==0)).sum())
        fn = int(((pred==0)&(mc==1)).sum())
        iou = tp/(tp+fp+fn+1e-6)
        tp_all+=tp; fp_all+=fp; fn_all+=fn
        rgb = raw[[2,1,0], T:T+IMG_SIZE, T:T+IMG_SIZE]
        rgb = (rgb/(np.percentile(rgb,98)+1e-6)*255).clip(0,255).astype(np.uint8).transpose(1,2,0)
        all_samples.append((iou,rgb,mc,pred,tp,fp,fn,mc.sum()/(IMG_SIZE**2)))

g_iou  = tp_all/(tp_all+fp_all+fn_all+1e-6)
g_prec = tp_all/(tp_all+fp_all+1e-6)
g_rec  = tp_all/(tp_all+fn_all+1e-6)
g_f1   = 2*g_prec*g_rec/(g_prec+g_rec+1e-6)
print(f'IoU={g_iou:.3f} F1={g_f1:.3f} Prec={g_prec:.3f} Rec={g_rec:.3f}')

vis = [s for s in all_samples if 0.08<=s[7]<=0.60]
vis.sort(key=lambda x: x[0])
n = len(vis)
picks = [('BEST',vis[int(n*.97)]),('MED #1',vis[int(n*.62)]),
         ('MED #2',vis[int(n*.38)]),('WORST',vis[int(n*.03)])]

def overlay(mask,pred):
    rgba = np.zeros((*mask.shape,4),dtype=np.float32)
    rgba[(pred==1)&(mask==1)] = [0.05,0.82,0.12,0.30]
    rgba[(pred==1)&(mask==0)] = [1.00,0.52,0.00,0.62]
    rgba[(pred==0)&(mask==1)] = [0.88,0.02,0.05,0.62]
    return rgba

fig = plt.figure(figsize=(15,7.5),facecolor='white')
fig.suptitle('Wildfire Burn Scar Detection from Sentinel-2 Satellite Imagery',
             fontsize=14,fontweight='bold',y=0.97)
fig.text(0.5,0.92,'PyTorch  |  Prithvi-100M (IBM/NASA)  |  6-channel Sentinel-2 L2A  |  dNBR labels  |  Corrientes, Argentina 2022',
         ha='center',fontsize=8.5,color='#444')
gs = gridspec.GridSpec(1,2,figure=fig,left=0.02,right=0.97,top=0.89,bottom=0.09,wspace=0.05,width_ratios=[2.1,1])
gs_p = gridspec.GridSpecFromSubplotSpec(2,2,subplot_spec=gs[0],hspace=0.03,wspace=0.03)
lcol = {'BEST':'#097a27','MED #1':'#b86e00','MED #2':'#b86e00','WORST':'#a30000'}
for (label,s),(r,c) in zip(picks,[(0,0),(0,1),(1,0),(1,1)]):
    iou,rgb,mc,pred = s[0],s[1],s[2],s[3]
    ax = fig.add_subplot(gs_p[r,c])
    ax.imshow(rgb,interpolation='bilinear'); ax.imshow(overlay(mc,pred),interpolation='none')
    ax.text(0.012,0.985,f'{label}  |  Patch IoU = {iou:.3f}',transform=ax.transAxes,
            va='top',ha='left',fontsize=8.5,fontweight='bold',color='white',
            bbox=dict(facecolor=lcol[label],edgecolor='none',pad=2.5,alpha=0.88))
    ax.axis('off')
fig.text(0.022,0.904,'Best / Median / Worst patches — validation set  (per-patch IoU)',fontsize=8,color='#555')
gs_r = gridspec.GridSpecFromSubplotSpec(2,1,subplot_spec=gs[1],hspace=0.44)
ax_c = fig.add_subplot(gs_r[0])
hist = history_v12 if 'history_v12' in dir() else history_v10
ep   = list(range(START_EPOCH if 'history_v12' in dir() else 1, START_EPOCH+len(hist['val_iou']) if 'history_v12' in dir() else len(hist['val_iou'])+1))
iou_arr = np.array(hist['val_iou'])
ax_c.plot(ep,hist['val_loss'],color='steelblue',lw=1.5,label='Val loss',alpha=0.85)
ax_c.plot(ep,iou_arr,color='#1aaa44',lw=1.5,label='Val IoU (burn scar)')
ax_c.axhline(iou_arr.max(),color='#1aaa44',ls='--',lw=1.2,alpha=0.7,label=f'Best: {iou_arr.max():.3f}')
ax_c.set_title('Training convergence  (epochs 41-80, Colab A100)',fontsize=8.5,fontweight='bold')
ax_c.set_xlabel('Epoch',fontsize=7.5); ax_c.tick_params(labelsize=7)
ax_c.grid(alpha=0.2); ax_c.legend(fontsize=7,frameon=False,loc='upper right'); ax_c.set_ylim(bottom=0.25)
ax_b = fig.add_subplot(gs_r[1])
names=['Recall','Precision','F1','IoU']; values=[g_rec,g_prec,g_f1,g_iou]
bars = ax_b.barh(names,values,color='#4a7fbf',height=0.48)
for bar,val in zip(bars,values):
    ax_b.text(val+0.012,bar.get_y()+bar.get_height()/2,f'{val:.3f}',
              va='center',fontsize=9.5,fontweight='bold',color='#111')
ax_b.set_xlim(0,1.0)
ax_b.set_title(f'Global validation metrics  (n={len(val_imgs):,} patches)',fontsize=8.5,fontweight='bold')
ax_b.tick_params(labelsize=8.5); ax_b.grid(axis='x',alpha=0.2)
ax_b.spines[['top','right','left']].set_visible(False)
fig.legend(handles=[
    mpatches.Patch(color=(0.05,0.82,0.12),label='True Positive'),
    mpatches.Patch(color=(1.00,0.52,0.00),label='False Positive'),
    mpatches.Patch(color=(0.88,0.02,0.05),label='False Negative'),
],loc='lower left',ncol=3,fontsize=8,frameon=False,bbox_to_anchor=(0.02,0.005))
fig.text(0.5,0.012,'Prithvi-EO-1.0-100M (IBM/NASA)  |  FPN decoder  |  Labels: dNBR > 0.10  |  Corrientes, Argentina 2022  |  ~900,000 ha burned',
         ha='center',fontsize=7.5,color='#555')
plt.savefig(RESULTS/'validation_overview.png',dpi=150,bbox_inches='tight',facecolor='white')
plt.show()
print(f'Saved: {RESULTS}/validation_overview.png')
